In [0]:
from pyspark.sql import SparkSession

In [0]:
spark=(
    SparkSession.builder
    .appName("Spark DataFrames")   # This is optional can be skipped
    .getOrCreate()
)

In [0]:
emp_data = [
    ("001","101","John Doe",30,"Male",50000,"2015-01-01"),
    ("002","101","Jane Smith",25,"Female",45000,"2016-02-15"),
    ("003","102","Bob Brown",35,"Male",55000,"2014-05-01"),
    ("004","102","Alice Lee",28,"Female",48000,"2017-09-30"),
    ("005","103","Jack Chan",40,"Male",60000,"2013-04-01"),
    ("006","103","Jill Wong",32,"Female",52000,"2018-07-01"),
    ("007","101","James Johnson",42,"Male",70000,"2012-03-15"),
    ("008","102","Kate Kim",29,"Female",51000,"2019-10-01"),
    ("009","103","Tom Tan",33,"Male",58000,"2016-06-01"),
    ("010","104","Lisa Lee",27,"Female",47000,"2018-08-01"),
    ("011",None,"David Park",38,"Male",65000,"2015-11-01"),   # bad/missing dept_id — intentional
    ("012","105","Susan Chen",31,"Female",54000,"2017-02-15"),
]
emp_schema = "employee_id string, department_id string, name string, age int, gender string, salary int, hire_date string"

In [0]:
emp=spark.createDataFrame(data=emp_data,schema=emp_schema)

In [0]:
emp.show()

In [0]:
emp.where(emp.department_id.isNull()).show()

In [0]:
emp.select('name').show()

In [0]:
# Get the name from the first row
first_name = emp.collect()[0]["name"]
print(first_name)

# Or using first()
print(emp.first()["name"])


In [0]:
emp.printSchema()

In [0]:
department_data=[
     ("1", "Data Science"),
     ("2", "Data Engineering"),
     ("3", "Accounts"),
     ("4", "Data Platform"),
     ("5", "Managed Services")
 ]

department_schema = "department_id string, department string"
department=spark.createDataFrame(data=department_data,schema=department_schema)
department.show()

# Join the two DataFrames
joined_df = emp.join(department, on="department_id", how="inner")

### Write our first Transformation (EMP salary > 50000)

**Key PySpark Functions:**
- **`where()` / `filter()`**: Filter rows based on a condition (both are aliases, work identically)
- **`col()`**: Reference a DataFrame column by name (needed for complex expressions)
- **`expr()`**: Write SQL-like expressions as strings (useful for dynamic or complex SQL logic)

In [0]:
from pyspark.sql.functions import col

## emp_sal_greater_than_50000= emp.where(col("salary") > 50000)
emp_filtered = emp.filter(emp.salary > 50000)
emp_filtered.show()

###  Write data as CSV output (ACTION)

In [0]:
## emp.coalesce(1).write.format("csv").mode("overwrite").save("/Volumes/workspace/default/raw_data/emp1_single")

### Using expr for select
### select employee_id as emp_id, name, cast(age as int) as age, salary from emp_filtered

In [0]:
emp_filtered.show()
emp_filtered.printSchema()

In [0]:
from pyspark.sql.functions import expr, col
emp_casted=emp_filtered.select(expr("employee_id as emp_id"), expr("name"), expr("age"), "salary")
emp_casted=emp_casted.orderBy(col("salary").desc())


In [0]:
emp_casted.show()

In [0]:
emp_casted.selectExpr("Emp_id","salary", "name as full_name", "age").show()

In [0]:
from pyspark.sql.functions import col

# Filter using the between() method
result_df = emp_casted.filter(col("salary").between(50000, 65000))

In [0]:
result_df.show()

Filter emp based on Age > 30
select emp_id, name, age, salary from emp_casted where age > 30

In [0]:
emp_casted.show()

In [0]:
emp.show()


In [0]:
from pyspark.sql.functions import col,expr

emp_selected=emp.select(col("employee_id"), expr("name as fullname"), emp.age, emp["salary"])
emp_selected.show()
emp.printSchema()


In [0]:
emp_age_casting=emp.select(col("employee_id"), expr("name as fullname"), emp.age.cast("string"), emp["salary"])
emp_age_casting.show()

In [0]:
emp_age_casting.printSchema()

In [0]:
emp.select(emp.name.alias("fullname")).show()

In [0]:
emp.select(emp.age.cast("Integer")).show()

In [0]:
emp_casted_ds=emp.selectExpr("employee_id as emp_id", "name", "cast(age as int) as AgeInNumbers", "salary","salary*0.25 as Bonus" ,"salary+(salary*0.25) as IncrementedSal")
emp_casted_ds.show()
